# Part 4: Face Recognition and Voiceprint Verification Models

**Owner:** Dorian Bitega Sheja (Part 4, model creation)
**Purpose:** Train, evaluate and save the face and voice models, and show that unknown people are rejected.
**Inputs:** `data/processed/image_features.csv`, `data/processed/audio_features.csv`
**Outputs:** `models/face_model.joblib`, `models/voice_model.joblib`, `outputs/metrics/face_metrics.json`, `outputs/metrics/voice_metrics.json`

All the training logic lives in `src/train_auth_models.py` so the notebook and the command line use the same code.

In [1]:
import os, sys, json, tempfile, random
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

# Work from the repository root, wherever the notebook was opened.
if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import train_auth_models as tam
print("Repository root folder:", ROOT.name)

Repository root folder: Formative-2-machine-learning


## 1. Face recognition model

### 1.1 The data
`image_features.csv` holds one row per original photo or augmented copy. `source_file` says which original photo a row came from. This matters: an augmented copy of a test photo must never be in the training set.

In [2]:
faces = pd.read_csv("data/processed/image_features.csv")
embedding_cols = [c for c in faces.columns if c.startswith("embedding_")]
print("rows:", len(faces), "| embedding features:", len(embedding_cols))
display(faces.groupby(["member_id", "augmentation"]).size().unstack(fill_value=0))

rows: 48 | embedding features: 512


augmentation,brightness_115,horizontal_flip,original,rotation_8deg
member_id,,,,
member01,3,3,3,3
member02,3,3,3,3
member03,3,3,3,3
member04,3,3,3,3


### 1.2 Training
Two candidate models are compared by cross-validated weighted F1 on the training rows only: Logistic Regression (baseline) and Random Forest. The better one is refitted and scored once on a held-out test set. The split is grouped by `source_file`, so no original photo is shared between train and test.

In [3]:
face_args = SimpleNamespace(data="data/processed/image_features.csv", label=None, group_col=None,
                            threshold=0.6, distance_factor=2.0)
tam.train("face", face_args)


=== FACE MODEL ===
Rows: 48 | label column: 'member_id' | group column: 'source_file'
Class counts:
member_id
member01    12
member02    12
member03    12
member04    12
Features used: 512 numeric, 0 categorical
Train rows: 32 | Test rows: 16
  logistic_regression: cross-validated weighted F1 = 1.000


  random_forest: cross-validated weighted F1 = 1.000
Chosen model: logistic_regression

TEST RESULTS  accuracy=1.000  F1=1.000  loss=0.012
              precision    recall  f1-score   support

    member01       1.00      1.00      1.00         4
    member02       1.00      1.00      1.00         4
    member03       1.00      1.00      1.00         4
    member04       1.00      1.00      1.00         4

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16

Saved models/face_model.joblib and outputs/metrics/face_metrics.json


In [4]:
face_metrics = json.load(open("outputs/metrics/face_metrics.json"))
pd.Series({k: v for k, v in face_metrics.items() if isinstance(v, (int, float, str))}).to_frame("value")

,value
model_type,logistic_regression
accuracy,1.0
f1_weighted,1.0
log_loss,0.012337
train_rows,32
test_rows,16


**Reading the result.** Accuracy, F1 and log loss are reported on photos the model never trained on. The test set is small (3 original photos per member), so a perfect score shows the pipeline works but should not be read as production-level security performance.

### 1.3 Rejecting an unknown face (leave-one-member-out)
A real unknown person is not in the data, so we simulate one: train on three members, then show the model the fourth member's photos. A correct model must **reject** them. This runs the same `train` and `predict_rows` code as the real model, inside a temporary folder so no real model file is overwritten.

In [5]:
def leave_one_out(df, kind, label="member_id"):
    rows = []
    for held_out in sorted(df[label].unique()):
        known = df[df[label] != held_out]
        unknown = df[df[label] == held_out]
        with tempfile.TemporaryDirectory() as tmp:
            csv = Path(tmp) / "known.csv"
            known.to_csv(csv, index=False)
            old = os.getcwd(); os.chdir(tmp)
            try:
                args = SimpleNamespace(data=str(csv), label=None, group_col=None,
                                       threshold=0.6, distance_factor=2.0)
                import contextlib, io
                with contextlib.redirect_stdout(io.StringIO()):
                    tam.train(kind, args)
                bundle = tam.joblib.load(Path(tmp) / "models" / f"{kind}_model.joblib")
            finally:
                os.chdir(old)
        results = tam.predict_rows(bundle, unknown)
        rejected = sum(1 for _, _, ok in results if not ok)
        rows.append({"unknown_member": held_out, "samples": len(results),
                     "rejected": rejected, "wrongly_accepted": len(results) - rejected})
    return pd.DataFrame(rows)

face_unknown = leave_one_out(faces, "face")
face_unknown

,unknown_member,samples,rejected,wrongly_accepted
0,member01,12,12,0
1,member02,12,12,0
2,member03,12,12,0
3,member04,12,12,0


In [6]:
far = face_unknown.wrongly_accepted.sum() / face_unknown.samples.sum()
print(f"False acceptance rate for unknown faces: {far:.1%}")

False acceptance rate for unknown faces: 0.0%


### 1.4 Authorized faces accepted (false rejection check)
Now the opposite check: photos of known members should be accepted by the saved model.

In [7]:
bundle = tam.joblib.load("models/face_model.joblib")
res = tam.predict_rows(bundle, faces)
pred = [r[0] for r in res]; ok = [r[2] for r in res]
print(f"Correct person identified: {np.mean(np.array(pred) == faces.member_id.values):.1%}")
print(f"Accepted: {np.mean(ok):.1%}  ->  false rejection rate on training photos: {1-np.mean(ok):.1%}")
print("Note: these rows include the training photos, so this is an upper bound; the honest score is the held-out test above.")

Correct person identified: 100.0%
Accepted: 100.0%  ->  false rejection rate on training photos: 0.0%
Note: these rows include the training photos, so this is an upper bound; the honest score is the held-out test above.


## 2. Voiceprint verification model

The voice model uses the same code on `audio_features.csv` (MFCC means/standard deviations, spectral roll-off, RMS energy, spectral centroid, zero-crossing rate). It will run as soon as the audio feature file is in the repository.

In [8]:
audio_path = Path("data/processed/audio_features.csv")
HAVE_AUDIO = audio_path.exists()
print("audio_features.csv found:" , HAVE_AUDIO)
if HAVE_AUDIO:
    voices = pd.read_csv(audio_path)
    display(voices.groupby(["member_id", "augmentation"]).size().unstack(fill_value=0))
    tam.train("voice", SimpleNamespace(data=str(audio_path), label=None, group_col=None,
                                       threshold=0.6, distance_factor=2.0))
else:
    print("Voice model not trained yet: data/processed/audio_features.csv is not in the repository.")

audio_features.csv found: False
Voice model not trained yet: data/processed/audio_features.csv is not in the repository.


In [9]:
if HAVE_AUDIO:
    voice_unknown = leave_one_out(voices, "voice")
    display(voice_unknown)
    print(f"False acceptance rate for unknown voices: {voice_unknown.wrongly_accepted.sum()/voice_unknown.samples.sum():.1%}")

### 2.1 Identity mismatch (face of one member, voice of another)
`verify_features(model_path, features, expected_member)` returns `authorized=True` only when the voice belongs to the same member the face model recognised. The CLI app (Member 4) uses it for the voice step.

In [10]:
if HAVE_AUDIO:
    a, b = sorted(voices.member_id.unique())[:2]
    row = voices[voices.member_id == a].iloc[0]
    voice_features = tam.joblib.load("models/voice_model.joblib")["features"]
    feats = {c: row[c] for c in voice_features}
    print(f"{a}'s voice, expected {a}:", tam.verify_features("models/voice_model.joblib", feats, a))
    print(f"{a}'s voice, expected {b}:", tam.verify_features("models/voice_model.joblib", feats, b))

## 3. Findings and limitations

- Both models are trained with a split that keeps augmented copies of one original together, which avoids leakage.
- Unknown people are rejected by a confidence threshold (0.6) and a distance check to each member's usual samples.
- Only three original photos per member and a few recordings are available, so the held-out test sets are tiny. Perfect scores demonstrate that the pipeline works; they are not evidence of production-level security.
- The threshold and distance factor are defaults. They should be tuned on validation data, not on the final unauthorized sample.

**Artifacts:** `models/face_model.joblib`, `models/voice_model.joblib`, `outputs/metrics/face_metrics.json`, `outputs/metrics/voice_metrics.json` (model and metric files are ignored by Git; re-create them by running this notebook).